In [1]:
import cv2
import os
import subprocess
import uuid
from yt_dlp import YoutubeDL

/Users/satyabratasatapathy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def download_youtube_video(url, output_path="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo+bestaudio/best',
        'outtmpl': output_path,
        'quiet': True,
        'merge_output_format': 'mp4',
        'ffmpeg_location': '/opt/homebrew/bin/ffmpeg'  # << REQUIRED on your setup
    }
    env = os.environ.copy()
    env['PATH'] = f"/opt/homebrew/bin:{env['PATH']}"  # Safety: extend PATH inside the subprocess

    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return output_path

In [3]:
def extract_screenshots(video_path, interval=0.5, output_dir="screenshots"):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps

    count = 0
    current_time = 0.0
    while current_time < duration:
        cap.set(cv2.CAP_PROP_POS_MSEC, current_time * 1000)
        success, frame = cap.read()
        if not success:
            break
        filename = os.path.join(output_dir, f"frame_{count:05d}.jpg")
        cv2.imwrite(filename, frame)
        count += 1
        current_time += interval

    cap.release()
    print(f"Extracted {count} screenshots into '{output_dir}'")


In [4]:
import subprocess

result = subprocess.run(['/opt/homebrew/bin/ffmpeg', '-version'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
built with Apple clang version 17.0.0 (clang-1700.0.13.3)
configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

In [5]:
if __name__ == "__main__":
    import uuid
    import os

    youtube_link = "https://youtu.be/cEh2Qip0-E4?si=66Z7HZ5at9nDDlah"
    temp_video_name = f"temp_{uuid.uuid4().hex}.mp4"

    print("Downloading video...")
    video_path = download_youtube_video(youtube_link, temp_video_name)

    print("Extracting screenshots every 0.5 seconds...")
    extract_screenshots(video_path, interval=0.5, output_dir="yt_screenshots")

    # Optional cleanup
    if os.path.exists(temp_video_name):
        os.remove(temp_video_name)
        print("Temporary video file removed.")


Extracting screenshots every 0.5 seconds...              
Extracted 418 screenshots into 'yt_screenshots'
Temporary video file removed.


In [6]:
def generate_csv(screenshot_dir, interval=0.5, csv_output_dir="csv_output", csv_name="screenshots.csv"):
    os.makedirs(csv_output_dir, exist_ok=True)
    files = sorted([f for f in os.listdir(screenshot_dir) if f.endswith(".jpg")])
    csv_path = os.path.join(csv_output_dir, csv_name)

    with open(csv_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "timestamp (seconds)"])
        for index, fname in enumerate(files):
            timestamp = round(index * interval, 2)
            writer.writerow([fname, timestamp])

    print(f"CSV file '{csv_name}' created in '{csv_output_dir}' with {len(files)} rows.")


In [7]:
import csv


In [8]:
print("Generating CSV...")
generate_csv(screenshot_dir="yt_screenshots", interval=0.5, csv_output_dir="yt_csv")


Generating CSV...
CSV file 'screenshots.csv' created in 'yt_csv' with 418 rows.
